# Day 6: Sentiment Analysis Pipeline
Objective: Use HuggingFace sentiment pipeline, inspect confidence
scores, test edge cases, and find where the model fails.

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print("Pipeline loaded.")
print("Model being used:", classifier.model.config._name_or_path)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Pipeline loaded.
Model being used: distilbert/distilbert-base-uncased-finetuned-sst-2-english


## 1. Basic Tests
Run on simple clear-cut sentences first.

In [2]:
basic_sentences = [
    "I love this product, it works amazingly well!",
    "This is the worst experience I have ever had.",
    "The package arrived on time.",
    "I am so happy with my purchase.",
    "I hate waiting in long queues."
]

print("=== BASIC SENTIMENT TESTS ===\n")
for sentence in basic_sentences:
    result = classifier(sentence)[0]
    print(f"Text  : {sentence}")
    print(f"Label : {result['label']}")
    print(f"Score : {result['score']:.4f}")
    print("-" * 60)

=== BASIC SENTIMENT TESTS ===

Text  : I love this product, it works amazingly well!
Label : POSITIVE
Score : 0.9999
------------------------------------------------------------
Text  : This is the worst experience I have ever had.
Label : NEGATIVE
Score : 0.9998
------------------------------------------------------------
Text  : The package arrived on time.
Label : POSITIVE
Score : 0.9947
------------------------------------------------------------
Text  : I am so happy with my purchase.
Label : POSITIVE
Score : 0.9999
------------------------------------------------------------
Text  : I hate waiting in long queues.
Label : NEGATIVE
Score : 0.9968
------------------------------------------------------------


## 2. Confidence Score Analysis
Score = how confident the model is (0 to 1).
High score = very confident. Low score = uncertain.

In [3]:
uncertain_sentences = [
    "It's okay, nothing special.",
    "The food was fine.",
    "I guess it works.",
    "Not bad, not great.",
    "Could be worse."
]

print("=== CONFIDENCE SCORE ANALYSIS ===\n")
for sentence in uncertain_sentences:
    result = classifier(sentence)[0]
    confidence = "HIGH" if result['score'] > 0.9 else "MEDIUM" if result['score'] > 0.7 else "LOW"
    print(f"Text       : {sentence}")
    print(f"Label      : {result['label']}")
    print(f"Score      : {result['score']:.4f} ({confidence} confidence)")
    print("-" * 60)

=== CONFIDENCE SCORE ANALYSIS ===

Text       : It's okay, nothing special.
Label      : NEGATIVE
Score      : 0.8190 (MEDIUM confidence)
------------------------------------------------------------
Text       : The food was fine.
Label      : POSITIVE
Score      : 0.9998 (HIGH confidence)
------------------------------------------------------------
Text       : I guess it works.
Label      : POSITIVE
Score      : 0.9995 (HIGH confidence)
------------------------------------------------------------
Text       : Not bad, not great.
Label      : NEGATIVE
Score      : 0.9700 (HIGH confidence)
------------------------------------------------------------
Text       : Could be worse.
Label      : NEGATIVE
Score      : 0.9996 (HIGH confidence)
------------------------------------------------------------


## 3. Sarcasm Tests
Can the model detect sarcasm? This is a known weakness.

In [4]:
sarcasm_sentences = [
    "Oh great, another Monday. Just what I needed.",
    "Wow, I absolutely love being stuck in traffic for 2 hours.",
    "Sure, because nothing makes me happier than doing extra work.",
    "Oh fantastic, the printer is broken again.",
    "Yeah right, this is totally the best day ever."
]

print("=== SARCASM TESTS ===\n")
for sentence in sarcasm_sentences:
    result = classifier(sentence)[0]
    print(f"Text  : {sentence}")
    print(f"Label : {result['label']} (Score: {result['score']:.4f})")
    print(f"Human : NEGATIVE (sarcasm)")
    correct = "✅ CORRECT" if result['label'] == "NEGATIVE" else "❌ WRONG"
    print(f"Result: {correct}")
    print("-" * 60)

=== SARCASM TESTS ===

Text  : Oh great, another Monday. Just what I needed.
Label : POSITIVE (Score: 0.9989)
Human : NEGATIVE (sarcasm)
Result: ❌ WRONG
------------------------------------------------------------
Text  : Wow, I absolutely love being stuck in traffic for 2 hours.
Label : POSITIVE (Score: 0.5683)
Human : NEGATIVE (sarcasm)
Result: ❌ WRONG
------------------------------------------------------------
Text  : Sure, because nothing makes me happier than doing extra work.
Label : NEGATIVE (Score: 0.9817)
Human : NEGATIVE (sarcasm)
Result: ✅ CORRECT
------------------------------------------------------------
Text  : Oh fantastic, the printer is broken again.
Label : NEGATIVE (Score: 0.9594)
Human : NEGATIVE (sarcasm)
Result: ✅ CORRECT
------------------------------------------------------------
Text  : Yeah right, this is totally the best day ever.
Label : POSITIVE (Score: 0.9999)
Human : NEGATIVE (sarcasm)
Result: ❌ WRONG
----------------------------------------------------

## 4. Edge Cases
Test unusual inputs — very short, very long, mixed sentiment.

In [5]:
edge_cases = [
    "Good.",                                          # very short
    "Bad.",                                           # very short negative
    "I love the design but hate the price.",          # mixed sentiment
    "The movie had great acting but a terrible plot.",# mixed sentiment
    "This product is not bad at all.",                # double negative
    "I don't hate it.",                               # implicit positive
    "AMAZING!!!",                                     # caps + punctuation
    "meh",                                            # slang
]

print("=== EDGE CASE TESTS ===\n")
for sentence in edge_cases:
    result = classifier(sentence)[0]
    print(f"Text  : {sentence}")
    print(f"Label : {result['label']} (Score: {result['score']:.4f})")
    print("-" * 60)

=== EDGE CASE TESTS ===

Text  : Good.
Label : POSITIVE (Score: 0.9998)
------------------------------------------------------------
Text  : Bad.
Label : NEGATIVE (Score: 0.9998)
------------------------------------------------------------
Text  : I love the design but hate the price.
Label : NEGATIVE (Score: 0.7346)
------------------------------------------------------------
Text  : The movie had great acting but a terrible plot.
Label : NEGATIVE (Score: 0.9968)
------------------------------------------------------------
Text  : This product is not bad at all.
Label : POSITIVE (Score: 0.9991)
------------------------------------------------------------
Text  : I don't hate it.
Label : POSITIVE (Score: 0.9947)
------------------------------------------------------------
Text  : AMAZING!!!
Label : POSITIVE (Score: 0.9999)
------------------------------------------------------------
Text  : meh
Label : POSITIVE (Score: 0.9790)
-----------------------------------------------------------

## 5. Batch Processing
Pass multiple sentences at once — more efficient than one by one.

In [6]:
batch = [
    "Absolutely brilliant product.",
    "Terrible customer service.",
    "It does the job.",
    "I would definitely recommend this.",
    "Never buying from here again."
]

print("=== BATCH PROCESSING ===\n")
results = classifier(batch)
for sentence, result in zip(batch, results):
    print(f"Text  : {sentence}")
    print(f"Label : {result['label']} | Score: {result['score']:.4f}")
    print("-" * 60)

=== BATCH PROCESSING ===

Text  : Absolutely brilliant product.
Label : POSITIVE | Score: 0.9999
------------------------------------------------------------
Text  : Terrible customer service.
Label : NEGATIVE | Score: 0.9997
------------------------------------------------------------
Text  : It does the job.
Label : POSITIVE | Score: 0.9998
------------------------------------------------------------
Text  : I would definitely recommend this.
Label : POSITIVE | Score: 0.9996
------------------------------------------------------------
Text  : Never buying from here again.
Label : NEGATIVE | Score: 0.8902
------------------------------------------------------------


## 6. Failure Analysis
Where does the model consistently fail?
Collect all wrong predictions in one place.

In [7]:
test_cases = [
    ("Oh great, another Monday.", "NEGATIVE"),
    ("I love the design but hate the price.", "MIXED"),
    ("This product is not bad at all.", "POSITIVE"),
    ("I don't hate it.", "POSITIVE"),
    ("It's okay, nothing special.", "NEUTRAL"),
    ("Could be worse.", "NEUTRAL"),
    ("meh", "NEGATIVE"),
]

print("=== FAILURE ANALYSIS ===\n")
correct = 0
wrong = 0

for text, expected in test_cases:
    result = classifier(text)[0]
    predicted = result['label']
    # Normalize — pipeline outputs POSITIVE/NEGATIVE
    is_correct = predicted == expected
    status = "✅" if is_correct else "❌"
    if is_correct:
        correct += 1
    else:
        wrong += 1
    print(f"{status} Text     : {text}")
    print(f"   Expected : {expected}")
    print(f"   Got      : {predicted} (score: {result['score']:.4f})")
    print("-" * 60)

print(f"\nAccuracy: {correct}/{len(test_cases)} correct")

=== FAILURE ANALYSIS ===

❌ Text     : Oh great, another Monday.
   Expected : NEGATIVE
   Got      : POSITIVE (score: 0.9982)
------------------------------------------------------------
❌ Text     : I love the design but hate the price.
   Expected : MIXED
   Got      : NEGATIVE (score: 0.7346)
------------------------------------------------------------
✅ Text     : This product is not bad at all.
   Expected : POSITIVE
   Got      : POSITIVE (score: 0.9991)
------------------------------------------------------------
✅ Text     : I don't hate it.
   Expected : POSITIVE
   Got      : POSITIVE (score: 0.9947)
------------------------------------------------------------
❌ Text     : It's okay, nothing special.
   Expected : NEUTRAL
   Got      : NEGATIVE (score: 0.8190)
------------------------------------------------------------
❌ Text     : Could be worse.
   Expected : NEUTRAL
   Got      : NEGATIVE (score: 0.9996)
------------------------------------------------------------
❌ Text

## 7. What model is running under the hood?

In [8]:
print("Model name    :", classifier.model.config._name_or_path)
print("Model type    :", classifier.model.config.model_type)
print("Num labels    :", classifier.model.config.num_labels)
print("Label mapping :", classifier.model.config.id2label)

Model name    : distilbert/distilbert-base-uncased-finetuned-sst-2-english
Model type    : distilbert
Num labels    : 2
Label mapping : {0: 'NEGATIVE', 1: 'POSITIVE'}


## 8. Observations & Notes

### What worked well
- Clear positive/negative sentences → high confidence (>0.99)
- Batch processing works identically to single sentence
- Very short inputs ("Good.", "Bad.") still classified correctly

### Where the model failed
- Sarcasm: "Oh great, another Monday" → classified POSITIVE
- Mixed sentiment: picks one side, ignores the other
- Double negatives: "not bad" sometimes misclassified
- Neutral sentences: forced into POSITIVE or NEGATIVE (no neutral class)

### Why no neutral class?
The default model (distilbert-base-uncased-finetuned-sst-2-english)
was trained on SST-2 dataset which only has POSITIVE and NEGATIVE.
No neutral class exists → model forces every input into one of two buckets.

### Key insight
Confidence score is NOT the same as accuracy.
Model can be 99% confident AND wrong (sarcasm case).
Always check score + label together, never trust score alone.

### What I want to explore next
- Load a model with NEUTRAL class (3-label sentiment)
- Try sentiment on longer text like product reviews